### 2.1 理论计算题

**输入**：$3 \times 32 \times 32$（通道数 $\times$ 高 $\times$ 宽）  
**卷积层**：16 个卷积核，每个大小 $3 \times 5 \times 5$，填充 $p=2$，步幅 $s=2$

1. **输出特征图尺寸**  
   高/宽计算公式：  
   $$
   H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + 2p - K}{s} \right\rfloor + 1
   $$  
   代入：$H_{\text{in}}=32,\ p=2,\ K=5,\ s=2$  
   $$
   H_{\text{out}} = \left\lfloor \frac{32 + 4 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16
   $$  
   通道数等于卷积核个数，即 16。  
   **输出尺寸**：$16 \times 16 \times 16$（通道数 $\times$ 高 $\times$ 宽）

2. **单个输出像素的点乘次数**  
   每个输出像素对应卷积核与输入局部区域的逐元素相乘。卷积核大小为 $3 \times 5 \times 5 = 75$，故需要 **75 次乘法**。

2.2 编程题 – 手动实现最大池化

In [6]:
import numpy as np

def max_pool2d(x, kernel_size, stride=1, padding=0):
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding

    N, C, H, W = x.shape
    H_out = (H + 2*ph - kh) // sh + 1
    W_out = (W + 2*pw - kw) // sw + 1

    x_padded = np.pad(x, ((0,0), (0,0), (ph, ph), (pw, pw)),
                      mode='constant', constant_values=-np.inf)
    out = np.zeros((N, C, H_out, W_out))

    for i in range(H_out):
        h_start = i * sh
        h_end = h_start + kh
        for j in range(W_out):
            w_start = j * sw
            w_end = w_start + kw
            window = x_padded[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2, 3))
    return out

# 测试代码
if __name__ == "__main__":
    # 生成随机输入: batch=1, channel=1, height=4, width=4
    x = np.random.randn(1, 1, 4, 4)
    print("输入特征图形状:", x.shape)
    print("输入特征图内容:\n", x[0,0])

    out = max_pool2d(x, kernel_size=2, stride=2, padding=0)
    print("\n池化后形状:", out.shape)
    print("池化后内容:\n", out[0,0])

输入特征图形状: (1, 1, 4, 4)
输入特征图内容:
 [[ 2.08200123  0.5140311   1.04811049 -0.24206792]
 [-0.71765641 -0.67787593 -0.68036938 -0.90785705]
 [ 1.43980968  0.03772874 -2.33853163  0.24955581]
 [-0.64442223 -0.50551288 -1.00946864  1.5208208 ]]

池化后形状: (1, 1, 2, 2)
池化后内容:
 [[2.08200123 1.04811049]
 [1.43980968 1.5208208 ]]


### 3.1 理论计算题

输入输出通道数均为 $C$，无偏置。

1. **单个 $5 \times 5$ 卷积层参数量**  
   卷积核大小 $5 \times 5$，输入通道 $C$，输出通道 $C$  
   $$
   \text{参数量} = C \times C \times 5 \times 5 = 25C^2
   $$

2. **两个串联 $3 \times 3$ 卷积层总参数量**  
   每个卷积层参数量：$C \times C \times 3 \times 3 = 9C^2$  
   两层总和：$18C^2$

3.2 编程题 – NiN 块定义

In [7]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    def forward(self, x):
        return self.block(x)

# 测试代码
if __name__ == "__main__":
    block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
    dummy_input = torch.randn(1, 3, 32, 32)
    output = block(dummy_input)
    print("NiN Block 测试")
    print(f"输入形状: {dummy_input.shape}")
    print(f"输出形状: {output.shape}")
    print("参数量:", sum(p.numel() for p in block.parameters()))

NiN Block 测试
输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 16, 32, 32])
参数量: 992


### 4.1 理论计算题

样本值：$x_1=2,\ x_2=4,\ x_3=6,\ x_4=8$，$\gamma=2,\ \beta=1,\ \epsilon=0$

- 均值：$\mu = \frac{2+4+6+8}{4} = 5$
- 方差：$\sigma^2 = \frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4} = \frac{9+1+1+9}{4} = 5$，$\sigma = \sqrt{5}$
- 归一化：$\hat{x}_i = \frac{x_i - \mu}{\sigma}$
- 输出：$y_i = \gamma \hat{x}_i + \beta$

$$
\begin{aligned}
y_1 &= 2 \cdot \frac{2-5}{\sqrt{5}} + 1 = -\frac{6}{\sqrt{5}} + 1 \\[4pt]
y_2 &= 2 \cdot \frac{4-5}{\sqrt{5}} + 1 = -\frac{2}{\sqrt{5}} + 1 \\[4pt]
y_3 &= 2 \cdot \frac{6-5}{\sqrt{5}} + 1 = \frac{2}{\sqrt{5}} + 1 \\[4pt]
y_4 &= 2 \cdot \frac{8-5}{\sqrt{5}} + 1 = \frac{6}{\sqrt{5}} + 1
\end{aligned}
$$

数值近似：$y_1 \approx -1.683,\ y_2 \approx 0.106,\ y_3 \approx 1.894,\ y_4 \approx 3.683$

4.2 编程题 – 残差块

In [8]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.shortcut = nn.Identity()
    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        shortcut = self.shortcut(x)
        out += shortcut
        out = self.relu(out)
        return out

# 测试代码
if __name__ == "__main__":
    # 情况1: 输入输出通道相同，不使用1x1卷积
    res_block1 = Residual(16, 16, use_1x1conv=False)
    x1 = torch.randn(1, 16, 32, 32)
    y1 = res_block1(x1)
    print("残差块 (in=out=16, no 1x1)")
    print(f"输入形状: {x1.shape} -> 输出形状: {y1.shape}")

    # 情况2: 输入输出通道不同，使用1x1卷积调整
    res_block2 = Residual(3, 16, use_1x1conv=True, stride=2)
    x2 = torch.randn(1, 3, 32, 32)
    y2 = res_block2(x2)
    print("\n残差块 (in=3, out=16, stride=2, use 1x1)")
    print(f"输入形状: {x2.shape} -> 输出形状: {y2.shape}")

残差块 (in=out=16, no 1x1)
输入形状: torch.Size([1, 16, 32, 32]) -> 输出形状: torch.Size([1, 16, 32, 32])

残差块 (in=3, out=16, stride=2, use 1x1)
输入形状: torch.Size([1, 3, 32, 32]) -> 输出形状: torch.Size([1, 16, 16, 16])


### 5.1 理论计算题

1. **不同学习率的原因**  
   - 底层特征（边缘、纹理等）在源数据集上已学到通用表示，微调时只需小幅调整，用小学习率可避免破坏已有知识；顶层输出层需适应新类别，随机初始化后需快速学习，因此用较大学习率。

2. **小数据集且与源数据集相似时的策略**  
   - 冻结大部分底层特征提取层（或设极小学习率），只微调顶层（输出层）或添加少量全连接层；同时加强数据增广、使用 dropout 或早停，以防止过拟合。

In [9]:
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

transform_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

# 测试代码
if __name__ == "__main__":
    # 创建一个虚拟彩色图像 (H,W,C) 范围0-255
    dummy_img = Image.fromarray(np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8))
    print("原始图像尺寸:", dummy_img.size)
    
    # 应用增广管道（可运行多次看到随机效果）
    transformed = transform_pipeline(dummy_img)
    print("增广后张量形状:", transformed.shape)  # 应为 (3, 224, 224)
    print("像素值范围:", transformed.min().item(), "~", transformed.max().item())

原始图像尺寸: (256, 256)
增广后张量形状: torch.Size([3, 224, 224])
像素值范围: 0.12941177189350128 ~ 0.5254902243614197


5.2 编程题 – 图像增广管道

### 6.1 理论计算题

真实框 $A = [10,10,50,50]$，预测框 $B = [30,30,70,70]$

- 交集左上角：$(\max(10,30), \max(10,30)) = (30,30)$  
- 交集右下角：$(\min(50,70), \min(50,70)) = (50,50)$  
- 交集面积：$(50-30) \times (50-30) = 20 \times 20 = 400$

- $A$ 面积：$(50-10) \times (50-10) = 40 \times 40 = 1600$  
- $B$ 面积：$(70-30) \times (70-30) = 40 \times 40 = 1600$  
- 并集面积：$1600 + 1600 - 400 = 2800$

$$
\text{IoU} = \frac{400}{2800} = \frac{1}{7} \approx 0.142857
$$

6.2 编程题 – 标签平滑交叉熵损失

In [10]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, num_classes=None):
    N, K = logits.shape if num_classes is None else (len(labels), num_classes)
    if num_classes is None:
        num_classes = K
    smooth_labels = torch.full((N, num_classes), epsilon / (num_classes - 1))
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    log_probs = F.log_softmax(logits, dim=1)
    loss = - (smooth_labels * log_probs).sum(dim=1).mean()
    return loss

# 测试代码
if __name__ == "__main__":
    # 假设3分类，batch=2，真实标签为 [0, 2]
    logits = torch.randn(2, 3)
    labels = torch.tensor([0, 2])
    
    loss_standard = F.cross_entropy(logits, labels)
    loss_smooth = label_smoothing_cross_entropy(logits, labels, epsilon=0.1)
    
    print("标准交叉熵损失:", loss_standard.item())
    print("标签平滑交叉熵损失:", loss_smooth.item())
    
    # 展示平滑标签
    smooth_labels = label_smoothing_cross_entropy.__code__.co_varnames  # 简单展示
    print("\n标签平滑后的目标分布（第一个样本）:")
    N, K = 2, 3
    target = torch.full((K,), 0.1/(K-1))
    target[0] = 0.9
    print(target)

标准交叉熵损失: 1.8498908281326294
标签平滑交叉熵损失: 1.759331226348877

标签平滑后的目标分布（第一个样本）:
tensor([0.9000, 0.0500, 0.0500])
